# AI6103 project: Counterfactual R-Drop on IMDB
**Runtime → Change runtime type → A100** (or L4 if you are short on compute units). Run the cells top to bottom.
Every run saves to Google Drive and is skipped once finished, so after a disconnect just re-run cells 1–3 and then the stage you were on.

### 1. Mount Drive and unpack the code
First upload `ai6103_code.zip` to `MyDrive/AI6103_project/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/AI6103_project'
import os
os.makedirs(PROJ, exist_ok=True)
%cd $PROJ
!unzip -o -q ai6103_code.zip -d .
!ls

### 2. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -r requirements.txt
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_bf16_supported())

### 3. Data: clone the CAD repo and build the cache (once, about 2 min)

In [ ]:
%cd $PROJ
![ -d data/cad ] || git clone -q https://github.com/acmi-lab/counterfactually-augmented-data data/cad
!python -c "import pickle,os; from data import build_data; os.makedirs('data',exist_ok=True); d=build_data('data/cad'); pickle.dump(d,open('data/cache.pkl','wb')); print(d['meta'], len(d['extra_orig']), len(d['cad_train_pairs']))"

### 4. (Optional, 1 min) Quick real-model test
One epoch of the full method. It should finish without errors and print a dev accuracy well above 0.5.

In [ ]:
!python train.py --condition cfrdrop --epochs 1 --out runs/check --seed 0

### 5. Stage 1: hyper-parameter sweep (13 dev-only runs, about 45 min on A100)

In [ ]:
!python run_all.py --stage sweep 2>&1 | grep -E '>>>|\"epoch\": 10|selected|Error|Traceback'
!cat runs/sweep/best.json

### 6. Stage 2: final runs (10 conditions × 3 seeds, about 2.5–3 h on A100)
Seed 1 for all conditions finishes first, so partial results are usable early.

In [ ]:
!python run_all.py --stage final 2>&1 | grep -E '>>>|cad_pair|Error|Traceback'

### 6b. Follow-up: margin-space CF term (about 35–40 min on L4)
Sweeps $\beta\in\{0.01, 0.1\}$ on seed 1, then runs 3 seeds of `cad_cfm`. Finished runs are skipped.

In [ ]:
!python run_all.py --stage extra --extra "--eval_bs 64" 2>&1 | grep -E '>>>|selected|cad_pair|Error|Traceback'

### 7. Analysis → `report/` (tables, figures, summary.json)
Download `report.zip` (about 5 MB, with per-run predictions) and send it to Claude.

In [ ]:
!python analyze.py
!python analyze_extra.py
!python make_figures.py
!zip -q -r report.zip report runs/sweep runs/final -x '*.pt'
!ls -la report.zip report/tables report/figs
from google.colab import files; files.download('report.zip')